# Creating Splits

In [8]:
# -------------------------
# 1️⃣ Importaciones y configuraciones
# -------------------------
import numpy as np
import json
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, GRU, TimeDistributed
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# -------------------------
# Cargar configuración desde config.json
# -------------------------
with open('../config.json', 'r') as f:
    config = json.load(f)

# Datos comunes
common_config = config['common']
ACTIONS = common_config['actions']
SEQUENCE_LENGTH = common_config['sequence_length']

# Datos específicos de train_model
train_config = config['train_model']
DATASET_PATH = train_config['dataset_path']
MODEL_EXPORT_NAME = train_config['model_export_name']
HAND_SELECTION = train_config['hand_selection']

# Configuración de arquitectura del modelo
model_arch = train_config['model_architecture']
MODEL_TYPE = model_arch['type']
LAYERS_CONFIG = model_arch['layers']

# Configuración de entrenamiento
training_config = train_config['training']
EPOCHS = training_config['epochs']
BATCH_SIZE = training_config['batch_size']
VALIDATION_SPLIT = training_config['validation_split']
OPTIMIZER = training_config['optimizer']
LOSS = training_config['loss']
METRICS = training_config['metrics']

# Configuración de split de datos
split_config = train_config['data_split']
TEST_SIZE = split_config['test_size']
STRATIFY = split_config['stratify']

# -------------------------
# 2️⃣ Cargar dataset desde archivo
# -------------------------
data = np.load(DATASET_PATH)
X = data['X']  # forma original: (num_samples, sequence_length, 21, 3)
y_labels = data['y']

# Seleccionar mano(s) según configuración
if HAND_SELECTION == 'left':
    # Solo mano izquierda: primeros 21 landmarks
    X = X[:, :, :21, :]
    print(f"Usando solo mano izquierda: {X.shape}")
elif HAND_SELECTION == 'right':
    # Solo mano derecha: últimos 21 landmarks
    X = X[:, :, 21:, :]
    print(f"Usando solo mano derecha: {X.shape}")
else:  # 'both'
    # Ambas manos: mantener todos los 42 landmarks
    print(f"Usando ambas manos: {X.shape}")

# Aplanar landmarks de cada frame: (21,3) -> (63) o (42,3) -> (126)
num_samples = X.shape[0]
X = X.reshape(num_samples, SEQUENCE_LENGTH, -1)  # ahora (num_samples, sequence_length, features)

# Convertir labels a one-hot
y = to_categorical(y_labels, num_classes=len(ACTIONS))

# -------------------------
# 3️⃣ Train / Test split
# -------------------------
stratify_param = y_labels if STRATIFY else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=stratify_param, random_state=42
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)

# -------------------------
# 4️⃣ Crear modelo
# -------------------------
model = Sequential()

# Construir capas según configuración
for i, layer_config in enumerate(LAYERS_CONFIG):
    layer_type = layer_config['type']
    
    if layer_type == 'TimeDistributed_Dense':
        units = layer_config['units']
        activation = layer_config['activation']
        if i == 0:
            # Primera capa necesita input_shape
            model.add(TimeDistributed(Dense(units, activation=activation), 
                                     input_shape=(SEQUENCE_LENGTH, X.shape[2])))
        else:
            model.add(TimeDistributed(Dense(units, activation=activation)))
    
    elif layer_type == 'Dropout':
        rate = layer_config['rate']
        model.add(Dropout(rate))
    
    elif layer_type == 'GRU':
        units = layer_config['units']
        return_sequences = layer_config.get('return_sequences', False)
        model.add(GRU(units, return_sequences=return_sequences))
    
    elif layer_type == 'Dense':
        units = layer_config.get('units', len(ACTIONS))
        activation = layer_config['activation']
        model.add(Dense(units, activation=activation))

model.compile(
    optimizer=OPTIMIZER,
    loss=LOSS,
    metrics=METRICS
)

model.summary()

# -------------------------
# 5️⃣ Entrenamiento
# -------------------------
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    validation_split=VALIDATION_SPLIT,
    batch_size=BATCH_SIZE,
    verbose=1
)

# Guardar modelo entrenado
model.save(MODEL_EXPORT_NAME)
print(f"Modelo guardado en {MODEL_EXPORT_NAME}")

# -------------------------
# 6️⃣ Evaluación / Métricas
# -------------------------
model = load_model(MODEL_EXPORT_NAME)

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print(f"Accuracy en test set: {acc:.4f}")
print("Matriz de confusión:")
print(cm)


Usando ambas manos: (3030, 20, 42, 3)
X_train: (2575, 20, 126) X_test: (455, 20, 126)
y_train: (2575, 6) y_test: (455, 6)


/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_7              │ (None, 20, 64)         │         8,128 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_14 (GRU)                    │ (None, 20, 64)         │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_15 (GRU)                    │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 60,326 (235.65 KB)

 Trainable params: 60,326 (235.65 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
258/258 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.7262 - loss: 0.6935 - val_accuracy: 0.9670 - val_loss: 0.1708
Epoch 2/15
258/258 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9714 - loss: 0.0893 - val_accuracy: 1.0000 - val_loss: 0.0065
Epoch 3/15
258/258 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9811 - loss: 0.0672 - val_accuracy: 1.0000 - val_loss: 0.0037
Epoch 4/15
258/258 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 1.0000 - loss: 0.0026 - val_accuracy: 1.0000 - val_loss: 0.0012
Epoch 5/15
258/258 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 1.0000 - loss: 0.0010 - val_accuracy: 1.0000 - val_loss: 6.3145e-04
Epoch 6/15
258/258 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9835 - loss: 0.0561 - val_accuracy: 1.0000 - val_loss: 0.0028
Epoch 7/15
258/258 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 1.0000 - loss: 0.0019 - val_accuracy: 1.0000 - val_loss: 8.1133e-04
Epoch 8/15
258/258 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 1.0000 - loss: 7.9932e-0

Modelo guardado en hand_gesture_model.h5
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step
Accuracy en test set: 1.0000
Matriz de confusión:
[[75  0  0  0  0  0]
 [ 0 76  0  0  0  0]
 [ 0  0 76  0  0  0]
 [ 0  0  0 76  0  0]
 [ 0  0  0  0 76  0]
 [ 0  0  0  0  0 76]]
